# Vincio: Evaluation as a CI Gate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohswedd/vincio/blob/main/examples/notebooks/04_evaluation.ipynb)

Treat quality like a build: a **golden dataset** of cases, a set of **metrics**, and
**gates** that turn scores into a pass/fail verdict your CI can block on. This notebook
mirrors `examples/07_evaluation_observability.py` and shows three ways to do it:

- **(A)** the one-line `evaluation(dataset, metrics=[...], gates={...}).run()` front door,
- **(B)** the verbose `EvalRunner(app, metrics=[...], gates={...}).run(dataset)` and how to
  inspect `report.gates` / `report.summary()`,
- **(C)** how a failing gate **blocks CI** — an `assert` over `report.gates`.

Everything runs **fully offline** on the bundled deterministic mock provider — no API
keys, no network.

In [ ]:
!pip install -q vincio

## Setup: the offline-first provider helper

Vincio's single entry point is `from vincio import ContextApp`. The `_provider()` helper
below returns the deterministic **mock** provider by default, so this notebook runs with
no keys and no network.

> **To run against a real model**, set `VINCIO_PROVIDER` (e.g. `openai`) and the matching
> API key (e.g. `OPENAI_API_KEY`); optionally set `VINCIO_MODEL`. The rest of the code is
> unchanged.

In [ ]:
import os
import tempfile
from pathlib import Path

from vincio import ContextApp, Dataset
from vincio.evals import EvalCase, EvalRunner
from vincio.providers import MockProvider, build_provider


def _provider():
    """Offline mock by default; a real provider when VINCIO_PROVIDER is set."""
    name = os.environ.get("VINCIO_PROVIDER", "mock")
    if name == "mock":
        return MockProvider(), "mock-1"
    return build_provider(name), os.environ.get("VINCIO_MODEL", "gpt-4o-mini")

## A grounded app under test

We point the app at a tiny docs corpus and turn on `answer_only_from_sources`, so the
grounding metrics (`groundedness`, `citation_accuracy`) actually have evidence to score
against. This is the app whose quality we want to gate in CI.

In [ ]:
# Lay down a small docs corpus in a temp folder (nothing touches your project tree).
docs_dir = Path(tempfile.mkdtemp(prefix="vincio_eval_")) / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)
(docs_dir / "refund_policy.md").write_text(
    "# Refund Policy\n\n"
    "Customers on the Pro plan may request refunds within 30 days.\n\n"
    "| Plan  | Window  | Fee |\n|-------|---------|-----|\n"
    "| Pro   | 30 days | $0  |\n| Basic | 14 days | $5  |\n",
    encoding="utf-8",
)


def build_app() -> ContextApp:
    """A grounded support app: answers only from the docs, with citations."""
    provider, model = _provider()
    app = ContextApp(name="refunds_qa", provider=provider, model=model)
    app.add_source("docs", path=str(docs_dir))
    # Grounding-only answering makes groundedness/citation_accuracy meaningful.
    app.set_policy("answer_only_from_sources", True)
    return app


app = build_app()
print("app ready:", app.name)

## The golden dataset

A `Dataset` is a named collection of `EvalCase`s. Each case carries an `input` (the
question), an `expected` answer (the reference for overlap-style metrics), and optional
`tags`/`difficulty`. This is your portable "golden set" — `dataset.save(path)` writes one
JSON object per line (JSONL) and `Dataset.load(path)` round-trips it.

In [ ]:
dataset = Dataset(
    name="refunds",
    cases=[
        EvalCase(
            id="c1",
            input="What is the refund window for the Pro plan?",
            expected="Refunds within 30 days for the Pro plan.",
            tags=["refund"],
        ),
        EvalCase(
            id="c2",
            input="How long is the Pro refund window?",
            expected="30 days",
            tags=["refund"],
            difficulty="easy",
        ),
    ],
)
print(f"{len(dataset)} cases in golden set '{dataset.name}'")

## (A) The one-line front door: `evaluation(...).run()`

`evaluation(...)` bundles the dataset, metrics, and gates into a single expression. It's
the ergonomic equivalent of registering evaluators and calling `app.evaluate(...)`. Here
we layer it onto our already-configured grounded `app` via `app=`, so the grounding
metrics stay meaningful.

The returned report carries `report.gates` (the pass/fail verdicts) and `report.summary()`
(per-metric aggregates). `evaluation` is `@experimental`, so you'll see a one-time
`VincioExperimentalWarning` — that's expected, not an error.

In [ ]:
from vincio.tasks import evaluation

report_a = evaluation(
    dataset,
    metrics=["groundedness", "citation_accuracy", "lexical_overlap"],
    gates={"groundedness": ">= 0.8", "citation_accuracy": ">= 0.8"},
    app=app,  # reuse the grounded app we built above
).run()

print("gates :", {k: v["passed"] for k, v in report_a.gates.items()})
print("means :", {k: v["mean"] for k, v in report_a.summary().items()})

## (B) The verbose form: `EvalRunner(...).run(dataset)`

`EvalRunner` is the full-control surface behind the front door: explicit metrics, gates,
and `concurrency`. It returns the same kind of report, so you can inspect `report.gates`
verdict-by-verdict, read `report.summary()` aggregates, or call `report.print_summary()`
for a human-readable table.

We add a couple of operational gates too — `p95_latency` and `cost` are first-class
metrics, so CI can fail on a latency or spend regression, not just a quality drop.

In [ ]:
runner = EvalRunner(
    app,
    metrics=["groundedness", "citation_accuracy", "lexical_overlap", "cost", "latency"],
    gates={
        "groundedness": ">= 0.8",
        "citation_accuracy": ">= 0.8",
        "p95_latency": "<= 60000",
    },
    concurrency=4,
)
report_b = runner.run(dataset, name="golden_run")

# A human-readable summary (the same one CI logs would show).
report_b.print_summary()

print("\nper-gate verdicts:")
for name, verdict in report_b.gates.items():
    status = "PASS" if verdict["passed"] else "FAIL"
    print(f"  [{status}] {name} {verdict['expression']} (value={verdict['value']})")

print("\nper-metric aggregates:")
for name, agg in report_b.summary().items():
    print(f"  {name:18s} mean={agg['mean']:<8} min={agg['min']:<8} max={agg['max']}")

## (C) Make CI block on a failing gate

Gates are only useful if they can fail the build. The pattern is: run the evaluation,
then `assert` that every gate passed — a failing `assert` is a non-zero exit code, which
is exactly what fails a CI step.

First we demonstrate a gate that *should* fail by setting an unrealistically strict
threshold (we require 99% lexical overlap, which our short reference answers don't reach),
then we show the green-build assertion you'd actually keep in your test suite.

In [ ]:
# A deliberately strict gate so we can watch CI catch a regression.
strict = EvalRunner(
    app,
    metrics=["lexical_overlap"],
    gates={"lexical_overlap": ">= 0.99"},  # intentionally too strict
).run(dataset, name="strict_run")

failed = {name: v for name, v in strict.gates.items() if not v["passed"]}
print("failing gates:", failed)


def assert_gates_pass(report) -> None:
    """Raise (non-zero exit) if any gate failed — the CI-blocking check."""
    broken = {n: v for n, v in report.gates.items() if not v["passed"]}
    assert not broken, f"quality gates failed: {broken}"


# The strict run *would* block CI:
try:
    assert_gates_pass(strict)
except AssertionError as exc:
    print("CI would FAIL ->", exc)

And the green-build assertion you keep in your test suite — the realistic gates from
section (B) pass, so CI stays green:

In [ ]:
# The realistic gates pass, so this is a no-op and CI proceeds.
assert_gates_pass(report_b)
print("all gates passed — CI is green")

## Recap

- A `Dataset` of `EvalCase`s is your portable golden set (`save`/`load` as JSONL).
- `evaluation(dataset, metrics=[...], gates={...}).run()` is the one-line front door;
  `EvalRunner(app, ...).run(dataset)` is the full-control form. Both return a report with
  `report.gates` and `report.summary()`.
- Gates cover quality (`groundedness`, `citation_accuracy`, `lexical_overlap`) **and**
  operations (`cost`, `p95_latency`).
- `assert` over `report.gates` turns a quality regression into a failing CI step.

Next: wire `assert_gates_pass(...)` into a `pytest` test and run it on every PR. See
`examples/07_evaluation_observability.py` for judges, red-teaming, regression baselines,
drift detection, and the trace store.